In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inizializzazione dei qubit

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Quando un circuito viene eseguito su un'unità di elaborazione quantistica (QPU) IBM&reg;, all'inizio del circuito viene tipicamente inserito un reset implicito per garantire che i qubit siano inizializzati a zero. Questo è controllato dal flag `init_qubits`, impostato come [opzione di esecuzione primitiva](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Tuttavia, il processo di reset non è perfetto e può portare a errori di preparazione dello stato. Per attenuare l'errore, il sistema inserisce anche un tempo di ritardo di ripetizione (o `rep_delay`) tra i circuiti. Ogni Backend ha un `rep_delay` predefinito diverso, ma di solito è più lungo di T1 per consentire all'ambiente di resettare i qubit. Il `rep_delay` predefinito può essere consultato eseguendo `backend.default_rep_delay`.

Tutte le QPU IBM utilizzano l'esecuzione a frequenza di ripetizione dinamica, che ti consente di modificare il `rep_delay` per ogni job. I circuiti che invii in un job primitivo vengono raggruppati insieme per l'esecuzione sulla QPU. Questi circuito vengono eseguiti iterando sui circuito per ogni shot richiesto; l'esecuzione avviene per colonne su una matrice di circuito e shot, come illustrato nella figura seguente.

![La prima colonna rappresenta lo shot 0. I circuiti vengono eseguiti in ordine da 0 a 3. La seconda colonna rappresenta lo shot 1. I circuiti vengono eseguiti in ordine da 0 a 3. Le colonne rimanenti seguono lo stesso schema. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matrice di esecuzione per colonne")

Poiché il `rep_delay` viene inserito tra i circuiti, ogni shot dell'esecuzione incontra questo ritardo. Pertanto, ridurre il `rep_delay` diminuisce il tempo totale di esecuzione sulla QPU, ma a scapito di un aumento del tasso di errore nella preparazione dello stato, come si può vedere nell'immagine seguente:

![Questa immagine mostra che man mano che il valore di `rep_delay` viene ridotto, il tasso di errore nella preparazione dello stato aumenta.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Ritardo di ripetizione versus tasso di errore")

Impostare sia `rep_delay=0` che `init_qubits=False` essenzialmente "fonde" i circuiti insieme, poiché i qubit inizieranno nello stato finale dello shot precedente.

Nota che, sebbene i circuiti in un job primitivo vengano raggruppati insieme per l'esecuzione sulla QPU, non vi è alcuna garanzia sull'ordine in cui vengono eseguiti i circuiti provenienti dai PUB. Pertanto, anche se invii `pubs=[pub1, pub2]`, non vi è alcuna garanzia che i circuiti di `pub1` vengano eseguiti prima di quelli di `pub2`. Non vi è nemmeno la garanzia che i circuiti dello stesso job vengano eseguiti come un singolo batch sulla QPU.

## Specificare rep_delay per un job primitivo

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Passi successivi
> **Tip:** - Prova un esempio nel tutorial [Quantum approximate optimization algorithm (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Consulta come [iniziare con le primitive.](./get-started-with-primitives)